# 09. Feature Extraction Test

- Goal: inspect rep-level and optional phase-level feature records.
- Docs: `docs_eng/pipeline/09_feature_extraction.md` / `docs/pipeline/09_feature_extraction.md`
- Inputs: Motion-attributed pose dataframe and exercise context from prior-stage cells.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Feature counts, namespaces, source fields, and availability notes.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.features import (
    PHASE_AWARE_FEATURE_FAMILIES,
    FeatureRecord,
    extract_rep_features,
    features_to_dataframe,
    summarize_phase_to_rep,
)
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig, ExerciseDefinitionConfig, FeaturesConfig,
    MotionAttributionConfig, NormalizationConfig, PhaseSegmentationConfig,
    PipelineConfig, ValidationConfig, run_pipeline,
)
from movement.segmentation import segment_phases
from movement.validation import run_basic_validation

print('imports OK')
print(f'PHASE_AWARE_FEATURE_FAMILIES: {PHASE_AWARE_FEATURE_FAMILIES}')

## Data Setup

Runs pipeline ①–⑦ including phase segmentation so that phase-level feature emission is active.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"
def_dir  = PROJECT_ROOT / "data/definitions/exercises"

df_raw = load_pose_csv(csv_path)
run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
ann_df       = load_annotation_csv(ann_path)
df_ann, _    = apply_annotation(df_raw, ann_df)
exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
df_norm, _   = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)
df_seg, _    = segment_phases(df_norm, exercise_def, fps_default=30.0)

print(f'phase labels in data: {df_seg["phase"].dropna().unique().tolist()}')
print(f'rep count: {df_seg["rep_id"].dropna().nunique()}')

## Direct extract_rep_features() Test

In [ ]:
records = extract_rep_features(df_seg, exercise_def)

rep_records   = [r for r in records if r.phase is None]
phase_records = [r for r in records if r.phase is not None]

print(f'total records  : {len(records)}')
print(f'rep-level      : {len(rep_records)}')
print(f'phase-level    : {len(phase_records)}')
print()
print('Sample rep-level record:')
print(f'  {rep_records[0]}')
if phase_records:
    print('Sample phase-level record:')
    print(f'  {phase_records[0]}')

## Check 1: FeatureRecord Fields

In [ ]:
for r in records:
    assert isinstance(r, FeatureRecord)
    assert r.feature_id,              f'feature_id empty: {r}'
    assert r.exercise_id == 'squat',  f'wrong exercise_id: {r.exercise_id}'
    assert r.value is not None,       f'value None: {r}'
    assert r.unit,                    f'unit empty: {r}'
    assert len(r.source_fields) > 0,  f'source_fields empty: {r.feature_id}'
print(f'PASS: all {len(records)} FeatureRecord fields valid')

## Check 2: Rep-level Records (phase=None)

In [ ]:
assert len(rep_records) > 0, 'no rep-level records produced'
rep_ids_with_features = {r.rep_id for r in rep_records if r.rep_id is not None}
sequence_level_records = [r for r in rep_records if r.rep_id is None]
all_rep_ids = set(df_seg.loc[df_seg['segment_type'] == 'rep', 'rep_id'].dropna().unique())
missing_rep_ids = all_rep_ids - rep_ids_with_features
print(f'reps in data: {all_rep_ids}')
print(f'reps with features: {rep_ids_with_features}')
print(f'sequence-level rep_records with rep_id=None: {len(sequence_level_records)}')
assert not missing_rep_ids, f'reps without features: {missing_rep_ids}'
print('PASS: annotated reps have rep-level feature records')


## Check 3: phase-level Records when Phase Column populated

In [ ]:
if phase_records:
    phase_labels = {r.phase for r in phase_records}
    print(f'phase labels in feature records: {phase_labels}')
    assert phase_labels.issubset({'Descent', 'Ascent', 'Bottom_Hold'})

    rep_ids_set = {r.feature_id for r in rep_records}
    phase_ids_set = {r.feature_id for r in phase_records}
    overlap = rep_ids_set & phase_ids_set
    print(f'rep-level IDs   : {len(rep_ids_set)}')
    print(f'phase-level IDs : {len(phase_ids_set)}')
    assert not overlap, f'feature_id namespace collision: {overlap}'
    print('PASS: phase-level feature records are present and namespaced')
else:
    print('NOTE: no phase-level records emitted; inspect 08 phase segmentation readiness for p01.')


## Check 4: phase-level source_fields Include Phase Segmentation irovenance

In [ ]:
ps_sources = [
    r for r in phase_records
    if any('phase_segmentation' in sf for sf in r.source_fields)
]
print(f'phase records with phase_segmentation in source_fields: {len(ps_sources)} / {len(phase_records)}')
if phase_records:
    assert len(ps_sources) == len(phase_records), 'all phase-level records must reference phase_segmentation in source_fields'
    print('PASS: all phase-level records carry phase_segmentation provenance')
else:
    print('NOTE: skipped because p01 has no phase-level feature records in this run.')


## Check 5: summarize_phase_to_rep()

In [ ]:
summary_records = summarize_phase_to_rep(records)
print(f'summary records: {len(summary_records)}')
for r in summary_records:
    print(f'  {r.feature_id}  rep={r.rep_id}  value={r.value:.4f}  phase={r.phase}')
if summary_records:
    assert all(r.phase is None for r in summary_records), 'summary records are rep-level (phase=None)'
    assert all('phase_rom_ratio' in r.feature_id for r in summary_records), 'expected Descent/Ascent ROM ratio summary'
    print('PASS: summarize_phase_to_rep() produces rep-level ratio records')
else:
    print('NOTE: no phase summary records because phase-level inputs are unavailable for this p01 run.')


## Check 6: features_to_dataframe()

In [ ]:
df_feat = features_to_dataframe(records)
required_cols = ['feature_id', 'exercise_id', 'rep_id', 'value', 'unit', 'source_fields', 'phase']
for col in required_cols:
    assert col in df_feat.columns, f'missing column: {col}'
print(f'PASS: features_to_dataframe() shape={df_feat.shape}')
print(df_feat[['feature_id', 'rep_id', 'phase', 'value', 'unit']].head(10).to_string())

## Check 7: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True)
cfg.motion_attribution  = MotionAttributionConfig(enabled=True)
cfg.features            = FeaturesConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS, ann_df=ann_df)

assert 'features' in pipe_report
n_feat = len(pipe_report['features'])
print(f'PASS: pipeline ⑨ features report: {n_feat} records')
print(f'steps executed: {list(pipe_report.keys())}')

## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
